# finance / household / costs

Annual growth rate of household costs to 2070 as three constants: low, base and high.

| Term | In This Notebook |
|---|---|
| **Intent** | Give one annual cost growth rate per scenario to inflate spending to 2070. Placeholder: the base is the mean observed cost growth, the low is the mean inflation and the high mirrors the premium of cost growth over inflation |
| **Project Flavour** | `static`, served by `mlserver` (`model.md` 3.2) |
| **Data Splits** | `training` ten years of inflation and cost growth, synthetic until the real series are loaded; `golden` the exact request that becomes `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, target, dataset names and signature names as constants (`model.md` 6.6).

In [ ]:
import json

from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "finance/household/costs"
MODEL_NAME = "finance_household_costs"
TARGETS = [{"kind": "served"}]
TRAINING = "household_costs_training_2026-11-05_2026-11-05"
GOLDEN = "household_costs_golden"
INPUT = "finance_household_costs_in"
OUTPUTS = ["finance_household_costs_growth_low_pct", "finance_household_costs_growth_base_pct", "finance_household_costs_growth_high_pct"]

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training` file only when it is absent (`model.md` 6.5). The source is synthetic until the real series is loaded, so no query runs.

In [ ]:
INFLATION_DEVIATIONS = [-1.2, 0.8, -0.4, 1.0, -0.6, 0.2, 0.6, -1.0, 0.4, 0.2]
GROWTH_DEVIATIONS = [0.5, -0.7, 0.3, 0.9, -0.4, -0.8, 0.6, 0.2, -0.3, -0.3]

training_pl, training_path = ensure_data(
    WORKSPACE,
    "training",
    TRAINING,
    lambda: {
        "year": list(range(2016, 2026)),
        "inflation_pct": [round(3.24 + deviation, 4) for deviation in INFLATION_DEVIATIONS],
        "cost_growth_pct": [round(4.83 + deviation, 4) for deviation in GROWTH_DEVIATIONS],
    },
)
print("training", training_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** take the mean inflation and the mean cost growth of `training` only (R6). The low scenario is the inflation mean, the base the cost growth mean and the high mirrors the premium of the base over the low, until the structural drivers are modelled (see the plan).

In [ ]:
INFLATION = round(float(training_pl["inflation_pct"].mean()), 2)
GROWTH = round(float(training_pl["cost_growth_pct"].mean()), 2)
CONSTANTS = dict(zip(OUTPUTS, [INFLATION, GROWTH, round(2 * GROWTH - INFLATION, 2)], strict=True))
print("constants", CONSTANTS)
assert list(CONSTANTS.values()) == [3.24, 4.83, 6.42], "the fit did not recover the long run means"

## Package

- **Stage:** 5, package.
- **Intent:** write the golden rows, build the signature, write a `static` project whose constants are the fitted values, verify it and replay the golden through the library loader (R1, R3).

In [ ]:
from workbench.model import build_signature, package_project, register_preview, signature_input, signature_output
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {INPUT: [1.0, None], "expect": ["ok", "error"]})

signature = build_signature(
    MODEL_NAME,
    [signature_input(INPUT, "none", "Unused input.")],
    [
        signature_output("finance_household_costs_growth_low_pct", "percent", "Annual growth rate of costs, percent per year, low scenario.", datatype="FP64"),
        signature_output("finance_household_costs_growth_base_pct", "percent", "Annual growth rate of costs, percent per year, base scenario.", datatype="FP64"),
        signature_output("finance_household_costs_growth_high_pct", "percent", "Annual growth rate of costs, percent per year, high scenario.", datatype="FP64"),
    ],
)
project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic annual series around the long run inflation and cost growth means, training split",
        [training_path, golden_path],
        "mean inflation, mean cost growth and the premium mirrored",
    ),
    constants=CONSTANTS,
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped loader")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)